In [7]:
import asyncio
import json
import re
import requests
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

# ==========================================
# 1. ENGLISH SCRAPER (USCCB - Playwright)
# ==========================================

SECTION_KEYWORDS = [
    (("reading 1", "first reading", "reading i"), "reading1"),
    (("responsorial psalm", "psalm"), "psalm"),
    (("reading 2", "second reading", "reading ii"), "reading2"),
    (("alleluia", "gospel acclamation", "verse before the gospel"), "alleluia"),
    (("gospel",), "gospel"),
]

def _looks_like_citation(text):
    return bool(re.match(
        r'^(cf\.\s*)?[1-3]?\s*[A-Z][a-zA-Z]+\.?\s+\d+(:\d+)?([-,]\s*\d+)*[a-z]?$',
        text.strip()
    ))

def _parse_usccb_html(html_content):
    eng_dict = {
        "feast_day": "Daily Readings",
        "reading1": [], 
        "psalm": [], 
        "reading2": [], 
        "alleluia": [], 
        "gospel": []
    }
    soup = BeautifulSoup(html_content, 'html.parser')

    if soup.title:
        title_text = soup.title.get_text(strip=True)
        clean_title = title_text.split('|')[0].split('-')[0].strip()
        if clean_title and "daily readings" not in clean_title.lower():
            eng_dict["feast_day"] = clean_title

    if eng_dict["feast_day"] == "Daily Readings":
        main_title = soup.select_one('.block-page-title h1, .content-header h1, h1.page-title')
        if main_title:
            t = main_title.get_text(strip=True)
            if t and not t.lower().startswith("reading"):
                eng_dict["feast_day"] = t

    current_section = None
    current_text = []

    def save_current_option():
        if not current_section or not current_text:
            return
        full_text = "\n".join(current_text).strip()
        if full_text and full_text not in eng_dict[current_section]:
            eng_dict[current_section].append(full_text)

    for tag in soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'div', 'a', 'span']):
        text = tag.get_text(separator=' ', strip=True)
        if not text:
            continue
        lower_text = text.lower().strip()

        if any(footer_trigger in lower_text for footer_trigger in [
            "email terms & privacy", "united states conference of catholic bishops", 
            "listen podcast", "en español", "view calendar", "get daily readings",
            "lectionary for mass", "privacy policy", "usccb", "subscribe"
        ]):
            if current_section == "gospel":
                save_current_option()
                current_section = None
            continue

        is_heading_tag = tag.name in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']

        if is_heading_tag and lower_text == "or":
            save_current_option()
            current_text = []
            continue

        if is_heading_tag or len(text) < 40:
            matched = None
            for keys, section in SECTION_KEYWORDS:
                if any(k in lower_text for k in keys):
                    matched = section
                    break
            if matched:
                save_current_option()
                current_section = matched
                current_text = []
                continue

        if current_section:
            if _looks_like_citation(text):
                continue
            if tag.name in ['p', 'div'] and not tag.find(['div', 'p']):
                if text not in current_text:
                    current_text.append(text)

    save_current_option()
    return eng_dict

async def scrape_usccb_async(date_str):
    url = f"https://bible.usccb.org/bible/readings/{date_str}.cfm"
    empty = {
        "feast_day": "Daily Readings",
        "reading1": [], "psalm": [], "reading2": [], "alleluia": [], "gospel": []
    }

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
            )
        )
        try:
            print(f"  -> Navigating to USCCB for {date_str}...")
            await page.goto(url, wait_until="domcontentloaded", timeout=20000)
            await page.wait_for_timeout(2000)
            html_content = await page.content()
        except Exception as e:
            print(f"  -> Error loading page: {e}")
            return empty
        finally:
            await browser.close()

    return _parse_usccb_html(html_content)


# ==========================================
# 2. VIETNAMESE SCRAPER (ThanhLinh.net - Requests)
# ==========================================

def scrape_thanhlinh(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'
    }
    
    viet_dict = {"reading1": [], "psalm": [], "reading2": [], "alleluia": [], "gospel": []}
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        response.encoding = 'utf-8' 
    except requests.exceptions.RequestException as e:
        print(f"Error fetching ThanhLinh data: {e}")
        return viet_dict
        
    soup = BeautifulSoup(response.content, 'html.parser')
    
    content_container = (
        soup.find('div', class_=re.compile(r'node__content|field--name-body|content-article', re.I)) 
        or soup.find('article') 
        or soup
    )
    
    current_section = None
    current_text = []

    def save_current_section():
        if current_section and current_text:
            cleaned_text = "\n".join(current_text).strip()
            if cleaned_text and cleaned_text not in viet_dict[current_section]:
                viet_dict[current_section].append(cleaned_text)

    for tag in content_container.find_all(['p', 'h3', 'h4', 'div', 'span']):
        text = tag.get_text(separator=' ', strip=True)
        if not text:
            continue
            
        lower_text = text.lower()
        
        if any(stop_word in lower_text for stop_word in [
            "mục đặc biệt", "xem tất cả", "phim ảnh", "thánh ca phụng vụ", "tủ sách"
        ]):
            if current_section:
                save_current_section()
                current_section = None
            continue

        matched_section = None
        if text.startswith("Bài Ðọc I:") or text.startswith("Bài Đọc I:"):
            matched_section = "reading1"
        elif text.startswith("Ðáp Ca:") or text.startswith("Đáp Ca:") or "thánh vịnh" in lower_text:
            matched_section = "psalm"
        elif text.startswith("Bài Ðọc II:") or text.startswith("Bài Đọc II:"):
            matched_section = "reading2"
        elif text.startswith("Alleluia:") or text.startswith("Tung Hô"):
            matched_section = "alleluia"
        elif text.startswith("Phúc Âm:") or text.startswith("Tin Mừng:"):
            matched_section = "gospel"

        if matched_section:
            save_current_section()
            current_section = matched_section
            current_text = []
            continue

        if current_section:
            if not any(noise in lower_text for noise in ["bấm vào đây", "đó là lời chúa", "thứ sáu"]):
                if text not in current_text:
                    current_text.append(text)

    save_current_section()
    return viet_dict


# ==========================================
# 3. DATA MAPPING (Strictly ENG or VIET)
# ==========================================

def prepare_template_data(user_inputs, scraped_eng, scraped_viet):
    final_data = {
        "feast_day": scraped_eng.get("feast_day"),
        "eng": {}, 
        "viet": {}
    }
    
    for section, choices in user_inputs.items():
        if section in ["date", "viet_url"]:
            continue
            
        lang = choices.get("lang", "eng")
        idx = choices.get("option_index", 0)
        
        if lang == "eng":
            if section in scraped_eng and len(scraped_eng[section]) > idx:
                final_data["eng"][section] = scraped_eng[section][idx]
            else:
                final_data["eng"][section] = None
        else:
            final_data["eng"][section] = None
                
        if lang == "viet":
            if section in scraped_viet and len(scraped_viet[section]) > idx:
                final_data["viet"][section] = scraped_viet[section][idx]
            else:
                final_data["viet"][section] = None
        else:
            final_data["viet"][section] = None
                
    return final_data


# ==========================================
# 4. DOCUMENT GENERATION (Single Column)
# ==========================================

def create_handout_docx(user_inputs, final_data, filename="mass_handout.docx"):
    doc = Document()
    
    for section in doc.sections:
        section.top_margin = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin = Inches(1)
        section.right_margin = Inches(1)
        
    p_title = doc.add_paragraph()
    p_title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run_org = p_title.add_run("ĐOÀN CHÚA HÀI ĐỒNG – TNTT\n")
    run_org.font.size = Pt(11)
    run_org.font.bold = True
    run_org.font.color.rgb = RGBColor(100, 100, 100)
    
    run_feast = p_title.add_run(f"{final_data.get('feast_day', 'Mass Readings')}")
    run_feast.font.size = Pt(16)
    run_feast.font.bold = True
    run_feast.font.color.rgb = RGBColor(20, 20, 20)
    
    doc.add_paragraph()
    
    sections_list = [
        ("reading1", "BÀI ĐỌC I / FIRST READING"),
        ("psalm", "ĐÁP CA / RESPONSORIAL PSALM"),
        ("reading2", "BÀI ĐỌC II / SECOND READING"),
        ("alleluia", "ALLELUIA"),
        ("gospel", "TIN MỪNG / GOSPEL")
    ]
    
    for key, label in sections_list:
        choice_conf = user_inputs.get(key, {"lang": "eng"})
        lang_choice = choice_conf.get("lang", "eng")
            
        eng_text = final_data["eng"].get(key)
        viet_text = final_data["viet"].get(key)
        
        text_to_print = viet_text if lang_choice == "viet" else eng_text
        if not text_to_print:
            continue
            
        h_para = doc.add_paragraph()
        run_h = h_para.add_run(label)
        run_h.font.bold = True
        run_h.font.size = Pt(12)
        run_h.font.color.rgb = RGBColor(0, 51, 102)
        
        content_para = doc.add_paragraph()
        run_content = content_para.add_run(text_to_print)
        run_content.font.size = Pt(10.5)
                
        doc.add_paragraph()

    doc.save(filename)
    print(f"Handout successfully saved as {filename}! Ready for Google Docs upload.")


# ==========================================
# 5. MAIN EXECUTION ENTRY POINT
# ==========================================

async def main():
    # 🎯 SELECT "eng" OR "viet" FOR EACH SECTION:
    app_inputs = {
        "date": "080726",  
        "viet_url": "https://thanhlinh.net/loi-chua-586",  
        "reading1": {"lang": "viet", "option_index": 0},
        "psalm":    {"lang": "eng",  "option_index": 0},
        "reading2": {"lang": "eng",  "option_index": 0},
        "alleluia": {"lang": "viet", "option_index": 0},
        "gospel":   {"lang": "viet", "option_index": 0}    
    }
    
    print(f"Scraping USCCB (English) for date {app_inputs['date']}...")
    eng_readings = await scrape_usccb_async(app_inputs["date"])
    
    print(f"Scraping ThanhLinh (Vietnamese) from {app_inputs['viet_url']}...")
    viet_readings = scrape_thanhlinh(app_inputs["viet_url"])
    
    print("Merging Data Based on User Inputs...")
    final_document_data = prepare_template_data(app_inputs, eng_readings, viet_readings)
    
    print("\nGenerating Word Document (.docx)...")
    create_handout_docx(app_inputs, final_document_data, "Mass_Handout_Selection.docx")

await main()

Scraping USCCB (English) for date 080726...
  -> Navigating to USCCB for 080726...
Scraping ThanhLinh (Vietnamese) from https://thanhlinh.net/loi-chua-586...
Merging Data Based on User Inputs...

Generating Word Document (.docx)...
Handout successfully saved as Mass_Handout_Selection.docx! Ready for Google Docs upload.


---
---

# Notes

1. Invalid date test case
2. Optional memorial different readings
3. Gospel has the optional memorial words (see 7 Aug 2026)